# 04 — NetworkX Mycorrhizal Connectivity Model

**Purpose:** Build a graph of mycorrhizal connectivity between individual trees
across Barcelona, identify disconnected fungal islands, quantify the network
leverage of each priority intervention zone, and simulate seasonal spread.

**Inputs:**
- `data/grid_trees.geojson` — 400 m grid with per-cell attributes (incl. `sealed_pct`)
- `data/scored_grid.geojson` — priority-zone scores from notebook 03
- `data/arbrat-viari.csv` + `data/arbrat-zona.csv` — raw tree inventory

**Outputs:**
- `data/network_nodes.geojson` — one row per tree with graph attributes
- `data/network_edges.geojson` — edges for the top-5 connected islands (sample)
- `data/network_islands.geojson` — one row per connected component
- `data/bridge_scores.csv` — top-15 zones ranked by network leverage

**Edge rules** (Jumpponen & Egerton-Warburton 2010; Teste et al. 2023):
- AM–AM: ≤ 15 m AND `sealed_pct < 0.7` in shared grid cell
- EM–EM: ≤ 35 m AND `sealed_pct < 0.7` in shared grid cell
- No AM–EM edges (different fungal guilds)

**Scale note:** The full inventory has 189 k trees.  For tractable runtime
the connectivity graph is built district-by-district for EM trees across
all districts, and for AM trees within a single demonstration district.
This is documented clearly at each relevant cell.

## 0 — Imports and configuration

In [1]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import geopandas as gpd
import networkx as nx
from scipy.spatial import cKDTree
from shapely.geometry import Point, MultiPoint, LineString
from shapely.ops import unary_union

np.random.seed(42)
pd.set_option("display.max_columns", 30)
warnings.filterwarnings("ignore", category=FutureWarning)

# ── Paths ────────────────────────────────────────────────────────────────────
REPO = Path("../")
DATA = REPO / "data"

VIARI_PATH         = DATA / "arbrat-viari.csv"
ZONA_PATH          = DATA / "arbrat-zona.csv"
GRID_TREES_PATH    = DATA / "grid_trees.geojson"
SCORED_GRID_PATH   = DATA / "scored_grid.geojson"

OUT_NODES     = DATA / "network_nodes.geojson"
OUT_EDGES     = DATA / "network_edges.geojson"
OUT_ISLANDS   = DATA / "network_islands.geojson"
OUT_BRIDGES   = DATA / "bridge_scores.csv"

# Edge-distance thresholds in metres (UTM31N)
AM_DISTANCE_M  = 15.0
EM_DISTANCE_M  = 35.0
SEAL_THRESHOLD = 0.7    # cells above this are barriers

print("Configuration set.")
for p, label in [
    (VIARI_PATH,       "arbrat-viari.csv"),
    (ZONA_PATH,        "arbrat-zona.csv"),
    (GRID_TREES_PATH,  "grid_trees.geojson"),
    (SCORED_GRID_PATH, "scored_grid.geojson"),
]:
    status = "FOUND" if p.exists() else "ABSENT (will synthesise)"
    print(f"  {label:30s}  {status}")

Configuration set.
  arbrat-viari.csv                FOUND
  arbrat-zona.csv                 FOUND
  grid_trees.geojson              FOUND
  scored_grid.geojson             FOUND


## 1 — Mycorrhizal-type lookup table

Derived from the top-20 species identified in notebook 01, cross-referenced
with FungalRoot (Soudzilovskaia et al. 2020) and published literature.
Species outside the lookup are dropped from the connectivity graph — their
fungal guild is genuinely unknown, not assumed AM.

In [2]:
# Top-20 species (notebook 01) with mycorrhizal-type annotations
# Sources: FungalRoot v2.0, Smith & Read (2008), Brundrett & Tedersoo (2018)
MYCO_LOOKUP = {
    # Species name                                  : myco_type
    "Platanus × acerifolia":                          "AM",
    "Celtis australis":                               "AM",
    "Tipuana tipu":                                   "AM",
    "Styphnolobium japonicum":                        "AM",
    "Melia azedarach":                                "AM",
    "Brachychiton populneus":                         "AM",
    "Jacaranda mimosifolia":                          "AM",
    "Pinus pinea":                                    "EM",
    "Ligustrum lucidum":                              "AM",
    "Pyrus calleryana 'Chanticleer'":                 "AM",
    "Ulmus pumila":                                   "AM",
    "Cercis siliquastrum":                            "AM",
    "Prunus cerasifera 'Pissardii'":                  "AM",
    "Cupressus sempervirens":                         "AM",
    "Citrus × aurantium":                             "AM",
    "Robinia pseudoacacia":                           "AM",
    "Pinus halepensis":                               "EM",
    "Quercus ilex":                                   "EM",
    "Magnolia grandiflora":                           "AM",
    "Grevillea robusta":                              "AM",
}

n_am = sum(1 for v in MYCO_LOOKUP.values() if v == "AM")
n_em = sum(1 for v in MYCO_LOOKUP.values() if v == "EM")
print(f"Lookup table: {len(MYCO_LOOKUP)} species — {n_am} AM, {n_em} EM")

Lookup table: 20 species — 17 AM, 3 EM


## 2 — Load and annotate tree inventory

We join the raw CSVs (or synthesise a representative sample), attach
`myco_type` from the lookup, and drop any tree without a known guild.

In [3]:
def load_tree_inventory() -> pd.DataFrame:
    """Load real CSVs if available, else synthesise a representative sample."""
    if VIARI_PATH.exists() and ZONA_PATH.exists():
        print("Loading real tree inventory from CSV files…")
        viari = pd.read_csv(VIARI_PATH, encoding="utf-8", low_memory=False)
        zona  = pd.read_csv(ZONA_PATH,  encoding="utf-8", low_memory=False)
        viari["source"] = "street"
        zona["source"]  = "park"
        df = pd.concat([viari, zona], ignore_index=True)
        print(f"  Loaded {len(df):,} trees ({len(viari):,} street + {len(zona):,} park)")
        return df
    else:
        print("WARNING: CSV files absent — synthesising 5,000-tree representative sample.")
        print("         Run notebook 01 data download steps to obtain real inventory.\n")

        DISTRICTS = [
            "CIUTAT VELLA", "EIXAMPLE", "SANTS - MONTJUÏC", "LES CORTS",
            "SARRIÀ - SANT GERVASI", "GRÀCIA", "HORTA - GUINARDÓ",
            "NOU BARRIS", "SANT ANDREU", "SANT MARTÍ",
        ]
        # BCN UTM31N bounding box (approx)
        x0, x1 = 419_000.0, 436_000.0
        y0, y1 = 4_575_000.0, 4_593_000.0
        n = 5_000
        rng = np.random.default_rng(42)

        # Weight species by approximate BCN abundance from notebook 01
        species_list = list(MYCO_LOOKUP.keys())
        weights = np.array([
            42828, 21304, 10748, 9931, 7248, 6323, 4916,
            4287, 4252, 3707, 3676, 3550, 3439, 2909,
            2828, 2824, 2549, 2307, 2153, 2134,
        ], dtype=float)
        weights /= weights.sum()

        df = pd.DataFrame({
            "codi":             [f"SYN_{i:05d}" for i in range(n)],
            "x_etrs89":         rng.uniform(x0, x1, n),
            "y_etrs89":         rng.uniform(y0, y1, n),
            "cat_nom_cientific": rng.choice(species_list, size=n, p=weights),
            "nom_districte":     rng.choice(DISTRICTS, size=n),
            "data_plantacio":    pd.NaT,
            "source":            rng.choice(["street", "park"], size=n),
        })
        return df


trees_raw = load_tree_inventory()

# Attach myco_type
trees_raw["myco_type"] = trees_raw["cat_nom_cientific"].map(MYCO_LOOKUP)
n_total = len(trees_raw)
trees_known = trees_raw[trees_raw["myco_type"].notna()].copy()
n_known = len(trees_known)
print(f"\nTotal trees:      {n_total:,}")
print(f"Known myco_type:  {n_known:,}  ({n_known/n_total*100:.1f}%)")
print(f"Dropped:          {n_total - n_known:,}  (species outside top-20 lookup)")
print("\nMyco type breakdown:")
print(trees_known["myco_type"].value_counts().to_string())

Loading real tree inventory from CSV files…
  Loaded 189,220 trees (145,486 street + 43,734 park)

Total trees:      189,220
Known myco_type:  143,948  (76.1%)
Dropped:          45,272  (species outside top-20 lookup)

Myco type breakdown:
myco_type
AM    134809
EM      9139


## 3 — Load grid and join sealed_pct to trees

Each tree is assigned the `sealed_pct` of its enclosing 400 m grid cell.
Trees in highly sealed cells (sealed_pct ≥ 0.7) cannot form edges —
their soil connection is physically severed.

In [4]:
def load_grid_with_sealing() -> gpd.GeoDataFrame:
    """
    Load scored_grid.geojson (preferred) or grid_trees.geojson.
    Both contain sealed_pct.  If neither exists, synthesise.
    """
    for path in [SCORED_GRID_PATH, GRID_TREES_PATH]:
        if path.exists():
            g = gpd.read_file(path)
            print(f"Grid loaded from: {path.name}  ({len(g):,} cells)")
            return g

    print("WARNING: No grid file found — synthesising sealed_pct grid.")
    from shapely.geometry import box
    x0, y0, x1, y1 = 419_000, 4_575_000, 436_000, 4_593_000
    cell_size = 400
    xs = np.arange(x0, x1, cell_size)
    ys = np.arange(y0, y1, cell_size)
    xx, yy = np.meshgrid(xs, ys)
    cells = [box(x, y, x + cell_size, y + cell_size)
             for x, y in zip(xx.ravel(), yy.ravel())]
    n = len(cells)
    rng = np.random.default_rng(42)
    g = gpd.GeoDataFrame(
        {
            "cell_id":    [f"CELL_{i:04d}" for i in range(n)],
            "sealed_pct": rng.beta(2, 5, size=n),
            "composite_B":rng.uniform(0, 1, size=n),
            "top15_scenario_B": False,
        },
        geometry=cells,
        crs="EPSG:25831",
    )
    # Flag 15 random cells as top-15 priorities
    top_idx = rng.choice(n, size=15, replace=False)
    g.loc[top_idx, "top15_scenario_B"] = True
    return g


grid = load_grid_with_sealing()
if grid.crs is None or grid.crs.to_epsg() != 25831:
    grid = grid.to_crs(epsg=25831)

# Build GeoDataFrame of trees in UTM31N
trees = trees_known.copy()
trees_gdf = gpd.GeoDataFrame(
    trees,
    geometry=gpd.points_from_xy(trees["x_etrs89"], trees["y_etrs89"]),
    crs="EPSG:25831",
)
trees_gdf["tree_id"] = trees_gdf["codi"].astype(str)

# Spatial join: attach grid cell attributes to each tree
grid_sub = grid[["geometry", "cell_id", "sealed_pct"]].copy()
if "sealed_pct" not in grid_sub.columns:
    grid_sub["sealed_pct"] = 0.3   # fallback default

trees_gdf = gpd.sjoin(trees_gdf, grid_sub, how="left", predicate="within")
trees_gdf["sealed_pct"] = trees_gdf["sealed_pct"].fillna(0.5)   # unjoined = medium seal
trees_gdf["cell_id"]    = trees_gdf["cell_id"].fillna("UNKNOWN")
trees_gdf = trees_gdf.drop(columns=["index_right"], errors="ignore")

print(f"\nTrees joined to grid: {len(trees_gdf):,}")
print(f"Sealed_pct stats — mean: {trees_gdf['sealed_pct'].mean():.3f}, "
      f"max: {trees_gdf['sealed_pct'].max():.3f}")
print(f"Trees in barrier cells (sealed_pct >= {SEAL_THRESHOLD}): "
      f"{(trees_gdf['sealed_pct'] >= SEAL_THRESHOLD).sum():,}")

Grid loaded from: scored_grid.geojson  (495 cells)

Trees joined to grid: 143,948
Sealed_pct stats — mean: 0.726, max: 0.894
Trees in barrier cells (sealed_pct >= 0.7): 98,371


## 4 — Graph construction

### Strategy

The full 189 k inventory would produce up to O(n²) distance evaluations.
We avoid this by:

1. Using `scipy.spatial.cKDTree.query_pairs` — O(n log n) nearest-neighbour
   search returning only pairs within the threshold radius.
2. Processing **EM trees** across all districts (≈ 9,000 trees — tractable).
3. Processing **AM trees** within the single highest-AM district as a
   demonstration (caps at ~10,000 trees in that district).

Both sub-graphs are merged into a single NetworkX `Graph`.
Edges are filtered by the sealing constraint — neither endpoint may be
in a cell with `sealed_pct ≥ 0.7`.

In [5]:
def build_subgraph(
    trees_subset: gpd.GeoDataFrame,
    myco_type: str,
    distance_m: float,
    seal_threshold: float = SEAL_THRESHOLD,
) -> tuple[nx.Graph, list]:
    """
    Build a NetworkX graph for a subset of trees of a single myco_type.

    Parameters
    ----------
    trees_subset : GeoDataFrame with columns x_etrs89, y_etrs89, tree_id,
                   sealed_pct, myco_type, nom_districte, cell_id
    myco_type    : "AM" or "EM"
    distance_m   : Maximum edge distance in metres
    seal_threshold : sealed_pct above which a cell is a barrier

    Returns
    -------
    G     : networkx.Graph
    edges : list of (tree_id_a, tree_id_b, distance)
    """
    if len(trees_subset) == 0:
        return nx.Graph(), []

    coords = trees_subset[["x_etrs89", "y_etrs89"]].values
    ids    = trees_subset["tree_id"].values
    sealed = trees_subset["sealed_pct"].values

    # Add nodes with attributes
    G = nx.Graph()
    for i, row in enumerate(trees_subset.itertuples(index=False)):
        G.add_node(
            row.tree_id,
            x=float(row.x_etrs89),
            y=float(row.y_etrs89),
            myco_type=myco_type,
            district=getattr(row, "nom_districte", "UNKNOWN"),
            cell_id=getattr(row, "cell_id", "UNKNOWN"),
            sealed_pct=float(row.sealed_pct),
        )

    # cKDTree nearest-neighbour search within radius
    tree_index = cKDTree(coords)
    pairs = tree_index.query_pairs(r=distance_m, output_type="ndarray")

    edges = []
    for i, j in pairs:
        # Sealing barrier check: both endpoints must be in non-barrier cells
        if sealed[i] >= seal_threshold or sealed[j] >= seal_threshold:
            continue
        dist = float(np.linalg.norm(coords[i] - coords[j]))
        G.add_edge(ids[i], ids[j], distance=dist, myco_type=myco_type)
        edges.append((ids[i], ids[j], dist))

    return G, edges


print("Graph construction functions defined.")

Graph construction functions defined.


In [6]:
# ── EM graph: all districts ─────────────────────────────────────────────────
em_trees = trees_gdf[trees_gdf["myco_type"] == "EM"].copy()
print(f"EM trees (all districts): {len(em_trees):,}")
G_em, em_edges = build_subgraph(em_trees, "EM", EM_DISTANCE_M)
print(f"  EM graph  — nodes: {G_em.number_of_nodes():,}, "
      f"edges: {G_em.number_of_edges():,}")

# ── AM graph: demonstration district (most AM trees) ───────────────────────
am_trees = trees_gdf[trees_gdf["myco_type"] == "AM"].copy()
if "nom_districte" in am_trees.columns:
    demo_district = am_trees["nom_districte"].value_counts().idxmax()
else:
    demo_district = "EIXAMPLE"
am_demo = am_trees[am_trees["nom_districte"] == demo_district].copy()
print(f"\nAM demonstration district: {demo_district} ({len(am_demo):,} trees)")
print("NOTE: AM graph built for demonstration district only to keep runtime tractable.")
print("      Scale to full city by iterating over all districts with same build_subgraph().")
G_am, am_edges = build_subgraph(am_demo, "AM", AM_DISTANCE_M)
print(f"  AM demo graph — nodes: {G_am.number_of_nodes():,}, "
      f"edges: {G_am.number_of_edges():,}")

# ── Merge into single graph ─────────────────────────────────────────────────
G = nx.compose(G_em, G_am)
all_edges = em_edges + am_edges
print(f"\nCombined graph — nodes: {G.number_of_nodes():,}, "
      f"edges: {G.number_of_edges():,}")

EM trees (all districts): 9,139
  EM graph  — nodes: 9,139, edges: 41,347

AM demonstration district: SANT MARTÍ (26,038 trees)
NOTE: AM graph built for demonstration district only to keep runtime tractable.
      Scale to full city by iterating over all districts with same build_subgraph().
  AM demo graph — nodes: 26,038, edges: 13,010

Combined graph — nodes: 35,177, edges: 54,357


## 5 — Connected components (fungal islands)

Each connected component represents a discrete fungal network island —
trees within a component are presumed able to exchange carbon and nutrients
via shared mycelium.  Components spanning multiple districts have the
highest conservation value as cross-district bridges.

In [7]:
components = list(nx.connected_components(G))
print(f"Total connected components: {len(components):,}")

comp_records = []
for comp_id, nodes in enumerate(components):
    node_list = list(nodes)
    n_nodes = len(node_list)
    xs = [G.nodes[nd]["x"] for nd in node_list if "x" in G.nodes[nd]]
    ys = [G.nodes[nd]["y"] for nd in node_list if "y" in G.nodes[nd]]
    cx = float(np.mean(xs)) if xs else np.nan
    cy = float(np.mean(ys)) if ys else np.nan
    myco_types = [G.nodes[nd].get("myco_type", "?") for nd in node_list]
    dominant_myco = pd.Series(myco_types).value_counts().idxmax() if myco_types else "?"
    districts = list({G.nodes[nd].get("district", "?") for nd in node_list})
    comp_records.append({
        "component_id":      comp_id,
        "node_count":        n_nodes,
        "dominant_myco_type":dominant_myco,
        "districts_spanned": len(districts),
        "district_names":    ", ".join(sorted(districts)[:5]),  # cap string length
        "centroid_x":        cx,
        "centroid_y":        cy,
    })

comp_df = pd.DataFrame(comp_records).sort_values("node_count", ascending=False)
print("\nTop 10 islands by node count:")
print(comp_df.head(10).to_string(index=False))

Total connected components: 25,508

Top 10 islands by node count:
 component_id  node_count dominant_myco_type  districts_spanned   district_names    centroid_x   centroid_y
         9755         552                 AM                  1       SANT MARTÍ 433406.349424 4.585791e+06
         8549         329                 AM                  1       SANT MARTÍ 433323.394809 4.585389e+06
         9769         307                 AM                  1       SANT MARTÍ 433404.813039 4.585591e+06
         8557         251                 AM                  1       SANT MARTÍ 433020.300797 4.585762e+06
          693         226                 EM                  1 HORTA - GUINARDÓ 428302.822553 4.587138e+06
         4076         208                 AM                  1       SANT MARTÍ 434150.846370 4.584546e+06
        13147         158                 AM                  1       SANT MARTÍ 433640.024772 4.583207e+06
          255         158                 EM                  1 HORTA 

In [8]:
top5 = comp_df.head(5)
print("Top 5 largest fungal islands:")
for _, row in top5.iterrows():
    print(f"  Island {int(row['component_id']):>4}  "
          f"nodes: {int(row['node_count']):>5}  "
          f"type: {row['dominant_myco_type']:>2}  "
          f"districts: {int(row['districts_spanned']):<2}  "
          f"centroid: ({row['centroid_x']:.0f}, {row['centroid_y']:.0f})")

top5_comp_ids = set(top5["component_id"].tolist())
# Map node → component_id
node_to_comp = {}
for comp_id, nodes in enumerate(components):
    for nd in nodes:
        node_to_comp[nd] = comp_id

Top 5 largest fungal islands:
  Island 9755  nodes:   552  type: AM  districts: 1   centroid: (433406, 4585791)
  Island 8549  nodes:   329  type: AM  districts: 1   centroid: (433323, 4585389)
  Island 9769  nodes:   307  type: AM  districts: 1   centroid: (433405, 4585591)
  Island 8557  nodes:   251  type: AM  districts: 1   centroid: (433020, 4585762)
  Island  693  nodes:   226  type: EM  districts: 1   centroid: (428303, 4587138)


## 6 — Bridge analysis: network leverage of priority zones

For each top-15 priority zone we ask: **how many additional tree pairs would
become connected if this cell's sealing barrier were removed?**

Method:
1. Take all trees in barrier cells within the zone.
2. Set those cells' `sealed_pct = 0` (simulating de-paving).
3. Recompute edges for those trees + their radius-neighbours.
4. Count new edges that cross previously-disconnected component boundaries.
5. Bridge score = number of new inter-component tree pairs connected.

In [9]:
def bridge_score_for_zone(
    zone_cell_id: str,
    trees_gdf: gpd.GeoDataFrame,
    node_to_comp: dict,
    G: nx.Graph,
) -> int:
    """
    Compute how many DISTINCT component pairs would be linked by removing
    the sealing barrier in the given zone cell.

    BUG-7 fix: previously this counted every potential new edge to a
    different-component neighbour, which double-counts and inflates the
    score whenever a blocked tree (component -1) reaches several neighbours
    in the same other component. We now collect the set of component IDs
    each blocked tree reaches and tally distinct unordered component pairs.

    Returns
    -------
    int : number of distinct (component_a, component_b) pairs that would
          become connected.
    """
    # Trees currently blocked in this zone's cell
    blocked = trees_gdf[
        (trees_gdf["cell_id"] == zone_cell_id) &
        (trees_gdf["sealed_pct"] >= SEAL_THRESHOLD)
    ].copy()

    if len(blocked) == 0:
        return 0

    # All trees within edge-distance radius of blocked trees
    blocked_coords = blocked[["x_etrs89", "y_etrs89"]].values

    # Build a local tree index for all trees in the graph
    all_graph_trees = trees_gdf[trees_gdf["tree_id"].isin(G.nodes())].copy()
    if len(all_graph_trees) == 0:
        return 0

    all_coords = all_graph_trees[["x_etrs89", "y_etrs89"]].values
    all_ids    = all_graph_trees["tree_id"].values
    all_sealed = all_graph_trees["sealed_pct"].values
    all_myco   = all_graph_trees["myco_type"].values

    kd = cKDTree(all_coords)

    new_component_pairs = set()
    for bi, (bx, by) in enumerate(blocked_coords):
        b_tree = blocked.iloc[bi]
        b_myco = b_tree["myco_type"]

        dist_thresh = AM_DISTANCE_M if b_myco == "AM" else EM_DISTANCE_M
        nbr_idx = kd.query_ball_point([bx, by], r=dist_thresh)

        reached_comps = set()
        for j in nbr_idx:
            if all_myco[j] != b_myco:           # no AM–EM edges
                continue
            if all_sealed[j] >= SEAL_THRESHOLD:  # neighbour also sealed
                continue
            n_id = all_ids[j]
            n_comp = node_to_comp.get(n_id, None)
            if n_comp is not None:
                reached_comps.add(n_comp)

        # Pair up every distinct component the blocked tree could bridge.
        comps_sorted = sorted(reached_comps)
        for a in range(len(comps_sorted)):
            for b in range(a + 1, len(comps_sorted)):
                new_component_pairs.add((comps_sorted[a], comps_sorted[b]))

    return len(new_component_pairs)


print("Bridge score function defined.")

Bridge score function defined.


In [10]:
# Identify top-15 zone cell_ids from scored_grid
if "top15_scenario_B" in grid.columns and "cell_id" in grid.columns:
    top15_cells = grid[grid["top15_scenario_B"] == True]["cell_id"].tolist()
else:
    # Fallback: pick 15 random cells
    top15_cells = grid["cell_id"].sample(15, random_state=42).tolist()
    print("WARNING: top15_scenario_B column not found — using random 15 cells for bridge demo.")

print(f"Computing bridge scores for {len(top15_cells)} priority zones…")
bridge_records = []

for i, cid in enumerate(top15_cells):
    score = bridge_score_for_zone(cid, trees_gdf, node_to_comp, G)
    bridge_records.append({"cell_id": cid, "bridge_score": score})
    if (i + 1) % 5 == 0 or (i + 1) == len(top15_cells):
        print(f"  Processed {i+1}/{len(top15_cells)} zones…")

bridge_df = pd.DataFrame(bridge_records).sort_values("bridge_score", ascending=False)
bridge_df["leverage_rank"] = range(1, len(bridge_df) + 1)

# Merge composite score for reference
if "composite_B" in grid.columns:
    bridge_df = bridge_df.merge(
        grid[["cell_id", "composite_B", "nom_districte", "intervention_type"]].drop_duplicates("cell_id"),
        on="cell_id", how="left",
    )

print("\nNetwork leverage ranking (top-15 zones by bridge score):")
print(bridge_df.to_string(index=False))

Computing bridge scores for 15 priority zones…
  Processed 5/15 zones…
  Processed 10/15 zones…
  Processed 15/15 zones…

Network leverage ranking (top-15 zones by bridge score):
 cell_id  bridge_score  leverage_rank  composite_B         nom_districte intervention_type
C014_016             0              1     0.794314      SANTS - MONTJUÏC         de-paving
C015_016             0              2     0.787718      SANTS - MONTJUÏC         de-paving
C015_019             0              3     0.760931             LES CORTS         de-paving
C016_010             0              4     0.796826      SANTS - MONTJUÏC         de-paving
C016_011             0              5     0.855370      SANTS - MONTJUÏC         de-paving
C020_016             0              6     0.797454      SANTS - MONTJUÏC         de-paving
C020_024             0              7     0.765726 SARRIÀ - SANT GERVASI         de-paving
C021_023             0              8     0.746666                GRÀCIA         de-paving
C0

## 7 — Simplified spread simulation

We simulate 5 seasons of mycorrhizal hyphal spread from three reference
source patches:

- **Collserola / Garraf** — large peri-urban woodland (EM dominated)
- **Ciutadella Park** — central urban park
- **Montjuïc** — hill park in Sants-Montjuïc district

**Spread rule:** In each season, every fungal island extends by 2 m in all
directions through non-sealed soil.  We approximate this by expanding the
convex hull of each island's tree points by `season × 2 m` and then
checking which trees fall inside the expanded hull.

**Scenarios compared:**
- Baseline: no intervention
- Intervention: top-3 bridge zones de-paved (sealed_pct set to 0)

In [11]:
# Reference patch bounding boxes in UTM31N (EPSG:25831)
SOURCE_PATCHES = {
    "Collserola": (416_000, 4_581_000, 425_000, 4_592_000),   # rough bbox
    "Ciutadella": (431_500, 4_582_500, 432_500, 4_583_500),
    "Montjuic":   (425_500, 4_577_000, 427_500, 4_579_500),
}

def trees_in_bbox(trees_gdf, xmin, ymin, xmax, ymax):
    """Return subset of trees within a bounding box."""
    return trees_gdf[
        (trees_gdf["x_etrs89"] >= xmin) & (trees_gdf["x_etrs89"] <= xmax) &
        (trees_gdf["y_etrs89"] >= ymin) & (trees_gdf["y_etrs89"] <= ymax)
    ]


source_nodes = set()
for patch_name, bbox in SOURCE_PATCHES.items():
    patch_trees = trees_in_bbox(trees_gdf, *bbox)
    ids = set(patch_trees["tree_id"].tolist())
    source_nodes |= ids
    print(f"Source patch {patch_name:15s}: {len(ids):>5} trees")

print(f"\nTotal source trees: {len(source_nodes):,}")

Source patch Collserola     :   161 trees
Source patch Ciutadella     :  2165 trees
Source patch Montjuic       :   138 trees

Total source trees: 2,464


> ## DEPRECATED — `simulate_spread` is not used by the v1 deliverable
>
> **Status (2026-05-10):** The `simulate_spread` function below is retained
> for reference but is **not consumed by the visualisation in notebook 05**.
> The independent spread-model audit (`outputs/spread-model-audit.md`)
> documented that:
>
> 1. The current frontier-BFS halts at season 0 on the corrected inputs.
>    Of ~2,464 source-patch trees, only 180 pass the `sealed_pct < 0.7`
>    filter, and none of those 180 has a non-barrier neighbour within
>    the 2 m / season spread radius. Five seasons run, zero growth.
> 2. The function accepts the NetworkX graph `G` as a parameter but
>    never reads it — it builds its own KDTree from `trees_gdf` and
>    bypasses the 15 m AM / 35 m EM edge thresholds entirely.
> 3. There is no AM vs EM differentiation, no cost-distance along the
>    path, and sealed-surface filtering is applied only at endpoints
>    (so a tree 1.9 m from another across a road counts as reached).
>
> **What the v1 deliverable uses instead:** notebook 05 renders a simpler,
> honestly-labelled *500 m connectivity neighbourhood* — a static buffer
> around the convex hull of each of the top-20 existing islands. That is
> not a projection and it does not depend on this function.
>
> A scientifically defensible rewrite would (a) traverse `G` with AM/EM
> rate differentiation, (b) compute cost-distance over a rasterised
> `sealed_pct` surface, (c) compare baseline vs intervention by running
> the BFS twice and rendering the difference. None of that ships in v1.


In [12]:
def simulate_spread(
    trees_gdf: gpd.GeoDataFrame,
    source_nodes: set,
    G: nx.Graph,
    n_seasons: int = 5,
    spread_m_per_season: float = 2.0,
    seal_threshold: float = SEAL_THRESHOLD,
    label: str = "baseline",
) -> dict:
    """
    Simulate seasonal spread of fungal networks from source patches.

    BUG-8 fix: the previous implementation issued a single radius query
    from each source node and grew the radius linearly with season — that
    is not propagation, it is a static buffer.  The 2 m/season "growth"
    was documented but never implemented because nothing was passed from
    season N to season N+1.

    The new implementation is frontier-based: each season, we extend the
    front by `spread_m_per_season` metres around every CURRENTLY-reached
    tree, filter by the sealing barrier, and accumulate newly-reached
    trees into the reached set. Over 5 seasons this produces genuine
    propagation rather than a series of independent radii.

    Returns dict keyed `season_0`..`season_n`, each a set of reached tree_ids.
    """
    # Build coordinate arrays for all non-barrier trees (the universe we
    # can ever reach). Trees in barrier cells are filtered out up-front.
    non_barrier = trees_gdf[trees_gdf["sealed_pct"] < seal_threshold].copy()
    if len(non_barrier) == 0:
        print(f"[{label}] No non-barrier trees — spread trivially 0.")
        return {f"season_{s}": set() for s in range(n_seasons + 1)}

    nb_coords = non_barrier[["x_etrs89", "y_etrs89"]].values
    nb_ids    = non_barrier["tree_id"].values
    id_to_idx = {tid: i for i, tid in enumerate(nb_ids)}

    kd = cKDTree(nb_coords)

    # Season 0: source trees that are themselves in non-barrier cells.
    reached = {tid for tid in source_nodes if tid in id_to_idx}
    history = {0: set(reached)}

    if not reached:
        print(f"[{label}] No source trees in non-barrier cells — spread trivially 0.")
        return {f"season_{s}": history.get(s, set()) for s in range(n_seasons + 1)}

    for season in range(1, n_seasons + 1):
        # Frontier = every reached tree (BFS-like expansion from the front).
        frontier_idx = [id_to_idx[tid] for tid in reached]
        frontier_coords = nb_coords[frontier_idx]

        radius = spread_m_per_season   # NB: per-season growth, not cumulative
        neighbour_idx_lists = kd.query_ball_point(frontier_coords, r=radius)

        new_reached = set()
        for nbr_list in neighbour_idx_lists:
            for j in nbr_list:
                tid = nb_ids[j]
                if tid in reached:
                    continue
                new_reached.add(tid)
        if not new_reached:
            # No further growth possible — record same set for remaining seasons
            for s in range(season, n_seasons + 1):
                history[s] = set(reached)
            break
        reached |= new_reached
        history[season] = set(reached)

    # Ensure we always emit n_seasons+1 keys even if growth halted early.
    for s in range(n_seasons + 1):
        history.setdefault(s, set(reached))

    print(f"[{label}] Spread result:")
    for s in [0, 1, 3, 5]:
        if s <= n_seasons:
            print(f"  Season {s}: {len(history[s]):,} trees reachable")
    return {f"season_{s}": history[s] for s in range(n_seasons + 1)}


print("Spread simulation function defined.")

Spread simulation function defined.


In [13]:
# ── Baseline spread ─────────────────────────────────────────────────────────
spread_baseline = simulate_spread(
    trees_gdf, source_nodes, G, n_seasons=5, label="baseline"
)

# ── Intervention: de-pave top-3 bridge zones ────────────────────────────────
top3_zones = bridge_df.head(3)["cell_id"].tolist()
print(f"\nTop-3 bridge intervention zones: {top3_zones}")

trees_intervened = trees_gdf.copy()
intervened_mask = trees_intervened["cell_id"].isin(top3_zones)
trees_intervened.loc[intervened_mask, "sealed_pct"] = 0.0
n_unblocked = intervened_mask.sum()
print(f"Trees unblocked by intervention: {n_unblocked}")

spread_intervention = simulate_spread(
    trees_intervened, source_nodes, G, n_seasons=5, label="top-3 intervention"
)

# ── Comparison ──────────────────────────────────────────────────────────────
print("\nSpread gain from top-3 bridge interventions:")
for s in [1, 3, 5]:
    baseline_n = len(spread_baseline[f"season_{s}"])
    interv_n   = len(spread_intervention[f"season_{s}"])
    gain = interv_n - baseline_n
    print(f"  Season {s}: baseline={baseline_n:,}  intervention={interv_n:,}  "
          f"gain=+{gain} trees")

[baseline] Spread result:
  Season 0: 180 trees reachable
  Season 1: 180 trees reachable
  Season 3: 180 trees reachable
  Season 5: 180 trees reachable

Top-3 bridge intervention zones: ['C014_016', 'C015_016', 'C015_019']
Trees unblocked by intervention: 538
[top-3 intervention] Spread result:
  Season 0: 180 trees reachable
  Season 1: 180 trees reachable
  Season 3: 180 trees reachable
  Season 5: 180 trees reachable

Spread gain from top-3 bridge interventions:
  Season 1: baseline=180  intervention=180  gain=+0 trees
  Season 3: baseline=180  intervention=180  gain=+0 trees
  Season 5: baseline=180  intervention=180  gain=+0 trees


## 8 — Save outputs

Four output files:

| File | Contents |
|------|----------|
| `network_nodes.geojson` | All trees in the graph with component_id, myco_type, district |
| `network_edges.geojson` | Edges for top-5 islands only (sample — full graph too large) |
| `network_islands.geojson` | One row per connected component with centroid geometry |
| `bridge_scores.csv` | Top-15 zones ranked by network leverage |

In [14]:
# ── network_nodes.geojson ───────────────────────────────────────────────────
node_records = []
for nd, attrs in G.nodes(data=True):
    node_records.append({
        "tree_id":    nd,
        "myco_type":  attrs.get("myco_type", "?"),
        "district":   attrs.get("district", "?"),
        "cell_id":    attrs.get("cell_id", "?"),
        "sealed_pct": attrs.get("sealed_pct", np.nan),
        "component_id": node_to_comp.get(nd, -1),
        "x": attrs.get("x", np.nan),
        "y": attrs.get("y", np.nan),
    })

nodes_df = pd.DataFrame(node_records)
nodes_gdf = gpd.GeoDataFrame(
    nodes_df,
    geometry=gpd.points_from_xy(nodes_df["x"], nodes_df["y"]),
    crs="EPSG:25831",
).drop(columns=["x", "y"])

nodes_gdf.to_file(OUT_NODES, driver="GeoJSON")
print(f"Saved: {OUT_NODES}  ({len(nodes_gdf):,} nodes)")

Saved: ..\data\network_nodes.geojson  (35,177 nodes)


In [15]:
# ── network_edges.geojson (top-5 islands only) ──────────────────────────────
# Collect edges whose both endpoints belong to a top-5 island
top5_nodes = set()
for cid in top5_comp_ids:
    for nd, attrs in G.nodes(data=True):
        if node_to_comp.get(nd) == cid:
            top5_nodes.add(nd)

edge_records = []
for u, v, edata in G.edges(data=True):
    if u in top5_nodes and v in top5_nodes:
        ux = G.nodes[u].get("x", np.nan)
        uy = G.nodes[u].get("y", np.nan)
        vx = G.nodes[v].get("x", np.nan)
        vy = G.nodes[v].get("y", np.nan)
        if not (np.isnan(ux) or np.isnan(uy) or np.isnan(vx) or np.isnan(vy)):
            edge_records.append({
                "source":     u,
                "target":     v,
                "myco_type":  edata.get("myco_type", "?"),
                "distance_m": round(edata.get("distance", 0.0), 2),
                "geometry":   LineString([(ux, uy), (vx, vy)]),
            })

if edge_records:
    edges_gdf = gpd.GeoDataFrame(edge_records, crs="EPSG:25831")
    edges_gdf.to_file(OUT_EDGES, driver="GeoJSON")
    print(f"Saved: {OUT_EDGES}  ({len(edges_gdf):,} edges for top-5 islands)")
else:
    print("No top-5 island edges to save — graph may be fully disconnected "
          "(typical with synthetic / small data).")
    # Write empty layer with correct schema
    empty_gdf = gpd.GeoDataFrame(
        columns=["source", "target", "myco_type", "distance_m"],
        geometry=gpd.GeoSeries([], crs="EPSG:25831"),
        crs="EPSG:25831",
    )
    empty_gdf.to_file(OUT_EDGES, driver="GeoJSON")
    print(f"Saved empty layer: {OUT_EDGES}")

Saved: ..\data\network_edges.geojson  (5,344 edges for top-5 islands)


In [16]:
# ── network_islands.geojson ─────────────────────────────────────────────────
# Build centroid Point geometry for each component
island_geoms = []
for row in comp_df.itertuples(index=False):
    cx = row.centroid_x
    cy = row.centroid_y
    if np.isnan(cx) or np.isnan(cy):
        island_geoms.append(None)
    else:
        island_geoms.append(Point(cx, cy))

islands_gdf = gpd.GeoDataFrame(
    comp_df.reset_index(drop=True),
    geometry=island_geoms,
    crs="EPSG:25831",
)
islands_gdf.to_file(OUT_ISLANDS, driver="GeoJSON")
print(f"Saved: {OUT_ISLANDS}  ({len(islands_gdf):,} components)")

Saved: ..\data\network_islands.geojson  (25,508 components)


In [17]:
# ── bridge_scores.csv ───────────────────────────────────────────────────────
bridge_df.to_csv(OUT_BRIDGES, index=False)
print(f"Saved: {OUT_BRIDGES}")
print()
print("Final bridge score table (network leverage ranking):")
print(bridge_df.to_string(index=False))

Saved: ..\data\bridge_scores.csv

Final bridge score table (network leverage ranking):
 cell_id  bridge_score  leverage_rank  composite_B         nom_districte intervention_type
C014_016             0              1     0.794314      SANTS - MONTJUÏC         de-paving
C015_016             0              2     0.787718      SANTS - MONTJUÏC         de-paving
C015_019             0              3     0.760931             LES CORTS         de-paving
C016_010             0              4     0.796826      SANTS - MONTJUÏC         de-paving
C016_011             0              5     0.855370      SANTS - MONTJUÏC         de-paving
C020_016             0              6     0.797454      SANTS - MONTJUÏC         de-paving
C020_024             0              7     0.765726 SARRIÀ - SANT GERVASI         de-paving
C021_023             0              8     0.746666                GRÀCIA         de-paving
C025_020             0              9     0.764214              EIXAMPLE         de-paving
C02

## 9 — Summary and key findings

This cell prints a concise summary for the session report.

In [18]:
n_nodes  = G.number_of_nodes()
n_edges  = G.number_of_edges()
n_comps  = len(components)
largest  = comp_df.iloc[0]["node_count"] if len(comp_df) > 0 else 0
top_bridge = bridge_df.iloc[0] if len(bridge_df) > 0 else None

print("=" * 60)
print("NOTEBOOK 04 — SUMMARY")
print("=" * 60)
print(f"Graph nodes (trees with known myco type): {n_nodes:,}")
print(f"Graph edges (potential mycelial links):   {n_edges:,}")
print(f"Connected components (fungal islands):    {n_comps:,}")
print(f"Largest island (node count):              {int(largest):,}")
print()
if top_bridge is not None:
    print(f"Top bridge zone: {top_bridge['cell_id']} "
          f"(bridge_score={int(top_bridge['bridge_score'])})")
print()
print("Outputs written:")
for path in [OUT_NODES, OUT_EDGES, OUT_ISLANDS, OUT_BRIDGES]:
    sz = f"{path.stat().st_size / 1024:.1f} KB" if path.exists() else "missing"
    print(f"  {path.name:<30s}  {sz}")
print()
print("NOTE: EM graph covers all districts. AM graph covers demonstration ")
print(f"district ({demo_district}) only.")
print("Scale to full city: iterate build_subgraph() over all districts.")

NOTEBOOK 04 — SUMMARY
Graph nodes (trees with known myco type): 35,177
Graph edges (potential mycelial links):   54,357
Connected components (fungal islands):    25,508
Largest island (node count):              552

Top bridge zone: C014_016 (bridge_score=0)

Outputs written:
  network_nodes.geojson           9804.8 KB
  network_edges.geojson           1439.0 KB
  network_islands.geojson         7774.9 KB
  bridge_scores.csv               0.9 KB

NOTE: EM graph covers all districts. AM graph covers demonstration 
district (SANT MARTÍ) only.
Scale to full city: iterate build_subgraph() over all districts.
